In [35]:
from erebus.individual_fit_results import IndividualFitResults
from erebus.joint_fit_results import JointFitResults
import numpy as np
import os
from uncertainties import ufloat
import yaml
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import cm
from matplotlib import colors as colours
from scipy.stats import norm
from erebus.utility.utils import bin_data
from erebus.utility.utils import get_eclipse_duration
from erebus.photometry_data import PhotometryData
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from astropy.time import Time
from erebus.systematics.frame_normalized_pca import perform_fn_pca_on_aperture
from scipy.optimize import curve_fit
from uncertainties import ufloat

plt.rcParams.update({
    'axes.formatter.use_mathtext': True,    
	'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'lines.linewidth': 1.0,
    'figure.dpi': 300,
})

single_column = (4, 3)
double_column = (8, 6)

def get_eclipse_duration_from_res(visit):
	return get_eclipse_duration(
		visit.results['inc'].nominal_value,
		visit.results['a_rstar'].nominal_value,
		visit.results['rp_rstar'].nominal_value,
		visit.results['p'].nominal_value
	) * 24

In [3]:
import numpy as np
from uncertainties import ufloat
from astropy.time import Time

In [60]:
early_eclipse = ufloat(-54, 2.4) / (60 * 24) 
late_eclipse = ufloat(-30, 4.9) / (60 * 24)

t0 = ufloat(10452.8997, 0.0012) + 2450000
P = ufloat(2.6162644, 0.0000026)
print(t0)

t = Time('2026-02-10 12:00:00', format='iso', scale='utc').jd
print(t - t0)
while t0.nominal_value < t:
	t0 += P
	
t_e = (t0 + P / 2) + early_eclipse
print(f"ecosw early: {(np.pi / 2.0) * (t_e - t0 - P/2) / P}")

t_e = (t0 + P / 2) + late_eclipse
print(f"ecosw late: {(np.pi / 2.0) * (t_e - t0 - P/2) / P}")

2460452.8997+/-0.0012
629.1003+/-0.0012
ecosw early: -0.0225+/-0.0010
ecosw late: -0.0125+/-0.0020


In [29]:
t0 = ufloat(10452.8997, 0.0012) + 2450000

xue_0 = ufloat(60864.4541, 0.0021) + 2400000.5
xue_1 = ufloat(60887.9993, 0.0050) + 2400000.5

xue_jf = ufloat(2460864.9537, 0.0014)

(((xue_jf - t0) % P) - (P/2.0)) * 24 * 60

/tmp/ipykernel_30799/4035109791.py:8: FutureWarning: AffineScalarFunc.__mod__() is deprecated. It will be removed in a future release.
  (((xue_jf - t0) % P) - (P/2.0)) * 24 * 60


-11.005919994620719+/-2.7199195765977175

In [33]:
ddt_ecosw = ufloat(-0.0103, 0.0015)
dt = ddt_ecosw * 2 * P / np.pi
print(dt * 24 * 60)

-25+/-4


In [34]:
# Updated eccentric global fits including RVs, transits, and first two eclipses from
# https://rockyworlds.stsci.edu/downloads/GJ3929b_JWST_DataAnalysisReport_Checkpoint01_%2020260306.pdf
# Gives ~18.3 minutes before the joint fit result above
print(dt * 24 * 60 - 18.3)

-43+/-4


In [36]:
# Verifying that my timing is accurate

In [37]:
# Load the photometry data
phot_files = glob("./Raw photometry/*.h5")
phot_data = [PhotometryData.load(f) for f in phot_files] 

In [38]:
print(phot_data[0].__dict__.keys())

dict_keys(['_cache_file', 'annulus_end', 'annulus_start', 'fits_file_location', 'radius', 'source_folder', 'visit_name', 'normalized_frames', 'raw_flux', 'time'])


In [62]:
phot_data[0].time.min() + 2400000.5

2460864.8329263083

In [42]:
import spelunker
spk = spelunker.load(pid=9235, obs_num=1)

Current working directory for spelunker: /mnt/c/Users/nicho/Research/GitHub/erebus/connors_et_al_2026/spelunker_outputs

Connecting with MAST GuideStar service...
	 Found 5 GS-FG _cal.fits file(s). Downloading...


In [43]:
print(spk.__dict__.keys())

dict_keys(['init_dir', 'directory', 'mast_api_token', 'fg_table', 'fg_array', 'fg_time', 'fg_flux', 'gaussfit_results', 'quickfit_results', 'pgram_results', 'object_properties', 'fontsize', 'pid', 'fg_datamodel', 'fg_timeseries', 'photometry_mask', 'obs_num', 'visit'])


In [65]:
t = Time(spk.fg_time[0], format='mjd')
print(t.jd)

2460864.829901736


In [69]:
((t.jd - t0.nominal_value) % P.nominal_value) / P.nominal_value

0.4497599397184162